# <center> <img src="../img/ITESOLogo.png" alt="ITESO" width="480" height="130"> </center>
# <center> **Departamento de Electrónica, Sistemas e Informática** </center>
---
## <center> **Big Data** </center>
---
## <center> **Lab 08** </center>
---
### <center> **Spring 2026** </center>
---
### <center> **Examples on Structured Streaming (files)** </center>
---

<center>Saul Razo Magallanes - 739974</center>
    <center>Ingeniería en Sistemas Computacionales</center>
    <center><strong>Profesor:</strong> Pablo Camarillo Ramírez</center>
    <center><strong>Fecha:</strong> 06/04/2026</center>

# Create SparkSession

In [21]:
from pcamarillor.spark_utils import SparkUtils

su = SparkUtils("Structured Streaming with Files",
                   master_url="spark://spark-master:7077")

su.spark

In [22]:
!ls /opt/spark/work-dir/data/streaming/logs

# Create a data stream from a local socket

### Connect Spark to the socket

In [ ]:
import pyspark.sql.functions as F
from pathlib import Path
import shutil

logs_schema = SparkUtils.generate_schema([("raw_line", "string")])

input_path = "/opt/spark/work-dir/data/streaming/logs/"

# Create the stream
logs_df = (su.spark.readStream
            .format("text")
            .option("maxFilesPerTrigger", 1) # Let's process one file at a time
            .schema(logs_schema)
            .load(input_path))

# Transform original dataframe
parsed_df = (
    logs_df
    .withColumn("parts",     F.split(F.col("raw_line"), r" \| "))
    .withColumn("timestamp", F.to_timestamp(F.col("parts")[0], "yyyy-MM-dd HH:mm:ss"))
    .withColumn("level",     F.trim(F.col("parts")[1]))
    .withColumn("message",   F.trim(F.col("parts")[2]))
    .withColumn("server",    F.trim(F.col("parts")[3]))
    .drop("parts", "raw_line")               # keep only the clean columns
    .filter(F.col("timestamp").isNotNull())  # skip malformed lines
)

# Let's create a summary
summary_df = (
    parsed_df
    .groupBy("server", "level")
    .count()
    .orderBy("server", "level")
)

# Clean checkpoint
checkpoint_path = "/opt/spark/work-dir/checkpoints/logs_checkpoint"
dir_path = Path(checkpoint_path)
if dir_path.exists() and dir_path.is_dir():
    shutil.rmtree(dir_path)

# Write stream in the destination
query_events = (
    parsed_df.writeStream
    .outputMode("append")        # append: show new rows only
    .format("console")
    .option("truncate", False)   # don't cut off long messages
    .option("numRows", 20)
    .option("checkpointLocation", checkpoint_path)
    .queryName("parsed_logs")
    .start()
)

query_summary = (
    summary_df.writeStream
    .outputMode("complete")
    .format("console")
    .option("truncate", False)
    .queryName("summary_logs")
    .start()
)

print("   Press Ctrl+C to stop.\n")
su.spark.streams.awaitAnyTermination()


26/04/07 02:37:10 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.
26/04/07 02:37:10 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-63381908-7e18-4c60-a817-6d601e1bafb1. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/04/07 02:37:10 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


   Press Ctrl+C to stop.



26/04/07 02:53:26 WARN CheckpointFileManager: Failed to rename temp file file:/opt/spark/work-dir/checkpoints/logs_checkpoint/sources/0/.0.c138cbd5-f1ed-4280-8cc5-e31984207631.tmp to file:/opt/spark/work-dir/checkpoints/logs_checkpoint/sources/0/0 because file exists
org.apache.hadoop.fs.FileAlreadyExistsException: rename destination file:/opt/spark/work-dir/checkpoints/logs_checkpoint/sources/0/0 already exists.
	at org.apache.hadoop.fs.FileSystem.rename(FileSystem.java:1699)
	at org.apache.hadoop.fs.DelegateToFileSystem.renameInternal(DelegateToFileSystem.java:211)
	at org.apache.hadoop.fs.AbstractFileSystem.renameInternal(AbstractFileSystem.java:896)
	at org.apache.hadoop.fs.AbstractFileSystem.rename(AbstractFileSystem.java:807)
	at org.apache.hadoop.fs.ChecksumFs.renameInternal(ChecksumFs.java:512)
	at org.apache.hadoop.fs.AbstractFileSystem.rename(AbstractFileSystem.java:807)
	at org.apache.hadoop.fs.FileContext.rename(FileContext.java:1044)
	at org.apache.spark.sql.execution.stre

-------------------------------------------
Batch: 0
-------------------------------------------
+-------------------+-----+------------------------------------+-------------+
|timestamp          |level|message                             |server       |
+-------------------+-----+------------------------------------+-------------+
|2026-04-07 02:53:26|INFO |Service restarted                   |server-node-4|
|2026-04-07 02:53:32|ERROR|Unhandled exception in worker thread|server-node-2|
|2026-04-07 02:53:42|WARN |Retry attempt 3 of 5                |server-node-2|
|2026-04-07 02:53:43|ERROR|500 Internal Server Error           |server-node-3|
|2026-04-07 02:53:48|INFO |User login successful               |server-node-3|
|2026-04-07 02:53:55|ERROR|Authentication failed               |server-node-5|
|2026-04-07 02:54:03|WARN |Response time above threshold       |server-node-3|
|2026-04-07 02:54:07|ERROR|Database connection timeout         |server-node-1|
|2026-04-07 02:54:10|ERROR|404 Not

-------------------------------------------
Batch: 0
-------------------------------------------
+-------------+-----+-----+
|server       |level|count|
+-------------+-----+-----+
|server-node-1|ERROR|1    |
|server-node-2|ERROR|1    |
|server-node-2|INFO |1    |
|server-node-2|WARN |1    |
|server-node-3|ERROR|2    |
|server-node-3|INFO |2    |
|server-node-3|WARN |1    |
|server-node-4|ERROR|1    |
|server-node-4|INFO |1    |
|server-node-5|ERROR|1    |
+-------------+-----+-----+

-------------------------------------------
Batch: 1
-------------------------------------------
+-------------------+-----+------------------------------------+-------------+
|timestamp          |level|message                             |server       |
+-------------------+-----+------------------------------------+-------------+
|2026-04-07 02:53:46|ERROR|Database connection timeout         |server-node-4|
|2026-04-07 02:53:55|WARN |Memory usage 90%                    |server-node-1|
|2026-04-07 02:54:

In [ ]:
su.spark.stop()